# 10c NSD test P-7: borrowing and the low-rank map in fMRI (prespecified, scored once)

**Kaggle:** CPU, Internet on. **Inputs:** the outputs of 10a and 10b.

Protocol P-7 (`docs/proposals/nsd_confirmation_protocol.md`). All 8 NSD people, each the new person in turn with 7
donors; the 872 images all saw are the test set; the fsaverage vertices common to all 8. Claims, Bonferroni over
three, one-sided, 7 df: C1 the low-rank map beats borrowing in encoding; C2 the combination beats borrowing in
decoding; C3 the combination beats the person's own ridge model at the same budget.

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '10c-nsdconfirm', 'cpu-confirm'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"K": 200, "ks_primary": [250, 500], "ks_descriptive": [100, 1000], "draws": [13, 14, 15], "folds": 5, "n_galleries": 4, "gallery_size": 200, "gallery_seed": 346, "delta_r_share": 0.05, "delta_top1": 2.0, "n_claims": 3, "alpha": 0.05, "wall_hours": 11.0, "roi_classes": ["prf-visualrois", "floc-faces", "floc-places", "floc-bodies", "floc-words"]}')
SUBJECTS = ['subj01', 'subj02', 'subj03', 'subj04', 'subj05', 'subj06', 'subj07', 'subj08']

import numpy as np
import pandas as pd
from scipy import stats
from cross_subject_encoding.eval import lowrank_map as lm
from cross_subject_encoding.eval import nsd_confirm as nc
from cross_subject_encoding.models.ridge import RidgePerOutput
from cross_subject_encoding.utils import config as C

if not isinstance(FIXED["K"], int):
    raise RuntimeError("K is not fixed: fill it in from 09q before the confirmation runs")
IN = Path("/kaggle/input")
conv = next(p.parent for p in IN.rglob("shared_train_nsd_ids.npy"))
shared = np.load(conv / "shared_train_nsd_ids.npy")
cols, common = {}, {}
for h in ("lh", "rh"):
    masks = [np.load(next((conv / f"{s}_roi").rglob(f"{h}.all-vertices_fsaverage_space.npy"))) for s in SUBJECTS]
    common[h], cl = nc.common_vertices(masks)
    cols[h] = dict(zip(SUBJECTS, cl))
V = sum(len(common[h]) for h in common)
ids, Y = {}, {}
for s in SUBJECTS:
    ids[s] = np.load(conv / f"{s}_train_nsd_ids.npy")
    Y[s] = np.concatenate([np.load(conv / f"{s}_{h}_fmri.npy", mmap_mode="r")[:, cols[h][s]] for h in ("lh", "rh")], 1)
noise = np.concatenate([np.concatenate([np.load(conv / f"{s}_{h}_noise_ceiling.npy").astype(np.float64)[cols[h][s]] for h in ("lh", "rh")]) for s in SUBJECTS])
DELTA_R = FIXED["delta_r_share"] * float(np.median(np.sqrt(np.clip(noise, 0, None) / 100)))
FEATS = {}
for name in ("alexnet", "dinov2"):
    f = np.load(next(IN.rglob(f"10b_{name}_kpca1000.npz")))
    FEATS[name] = (f["z"].astype(np.float64), {int(i): j for j, i in enumerate(f["nsd_ids"])})
feats = lambda name, nsd_ids: FEATS[name][0][[FEATS[name][1][int(i)] for i in nsd_ids]]
test_ids = np.sort(shared)
rows_of = {s: {int(i): j for j, i in enumerate(ids[s])} for s in SUBJECTS}
test_rows = {s: np.array([rows_of[s][int(i)] for i in test_ids]) for s in SUBJECTS}
pool_rows = {s: np.setdiff1d(np.arange(len(ids[s])), test_rows[s]) for s in SUBJECTS}
GALS = nc.galleries(len(test_ids), FIXED["n_galleries"], FIXED["gallery_size"], FIXED["gallery_seed"])
rois = {}
for cls in FIXED["roi_classes"]:
    for s in SUBJECTS:
        lab = np.concatenate([np.load(next((conv / f"{s}_roi").rglob(f"{h}.{cls}_challenge_space.npy")))[cols[h][s]] for h in ("lh", "rh")])
        rois[(s, cls)] = lab
print(f"{V} common vertices; delta_r {DELTA_R:.4f}; {len(test_ids)} test images; pools", {s: len(pool_rows[s]) for s in SUBJECTS},
      "|", f"{(time.monotonic() - T0) / 60:.1f} min")

In [ ]:
def ols_coef(x, y):
    # the paper's OLS (intercept, normal equations), per output, in float64
    xa = np.concatenate([np.ones((len(x), 1)), x], 1)
    return np.linalg.solve(xa.T @ xa, xa.T @ np.asarray(y, np.float64)).astype(np.float32)
ols_pred = lambda beta, x: (np.concatenate([np.ones((len(x), 1)), x], 1) @ beta.astype(np.float64))
BETA = {}
for name in ("alexnet", "dinov2"):
    for s in SUBJECTS:
        BETA[(name, s)] = ols_coef(feats(name, ids[s][pool_rows[s]]), Y[s][pool_rows[s]])
    print(name, "encoders fitted |", f"{(time.monotonic() - T0) / 60:.1f} min")

In [ ]:
def oof(fit_predict, k, seed):
    split = np.array_split(np.random.default_rng(seed).permutation(k), FIXED["folds"])
    out = None
    for held in split:
        keep = np.setdiff1d(np.arange(k), held)
        pred = fit_predict(keep, held)
        if out is None:
            out = np.empty((k, pred.shape[1]))
        out[held] = pred
    return out

def scores(pred, meas, centred=False):
    return {"enc_r": nc.encoding_r(pred, meas), "top1": float(np.mean([nc.identify_top1(pred[g], meas[g], centred) for g in GALS]))}

rows, roi_rows, fc_rows = [], [], []
stopped = None
for p, s in enumerate(SUBJECTS, start=1):
    t = time.monotonic()
    donors = [d for d in SUBJECTS if d != s]
    meas = np.asarray(Y[s][test_rows[s]], np.float64)
    y_pool = np.asarray(Y[s][pool_rows[s]], np.float32)
    X = {name: (feats(name, ids[s][pool_rows[s]]), feats(name, test_ids)) for name in ("alexnet", "dinov2")}
    def donor_mean(name, x):  # a running sum: 7 full prediction arrays would not fit in memory at once
        acc = np.zeros((len(x), V), np.float32)
        for d in donors:
            acc += ols_pred(BETA[(name, d)], x).astype(np.float32)
        return acc / len(donors)
    avg = {name: (donor_mean(name, X[name][0]), donor_mean(name, X[name][1]).astype(np.float64)) for name in ("alexnet", "dinov2")}
    basis = {name: lm.prediction_basis(avg[name][0], FIXED["K"]) for name in ("alexnet", "dinov2")}
    own_full = ols_pred(ols_coef(X["alexnet"][0], y_pool), X["alexnet"][1])
    def add(method, k, d, pred, centred=False, extra=None):
        rows.append({"subject": s, "method": method, "k": k, "draw": d, "centred": centred, **scores(pred, meas, centred), **(extra or {})})
    add("avg", 0, "-", avg["alexnet"][1], centred=True)
    add("avg", 0, "-", avg["alexnet"][1])
    add("own_full", "all", "-", own_full)
    for d in FIXED["draws"]:
        order = np.random.default_rng([d, p]).permutation(len(pool_rows[s]))
        for k in FIXED["ks_primary"] + FIXED["ks_descriptive"]:
            idx = order[:k]
            y_k, xk = y_pool[idx].astype(np.float64), X["alexnet"][0][idx]
            parts = {}
            for name in ("alexnet", "dinov2"):
                a_k, a_t = avg[name][0][idx].astype(np.float64), avg[name][1]
                mean, B = basis[name]
                model = lm.fit_lowrank_map(a_k, y_k, mean, B, seed=d)
                m_test = lm.apply_lowrank_map(model, a_t, mean, B)
                m_oof = oof(lambda tr, he: lm.apply_lowrank_map(lm.fit_lowrank_map(a_k[tr], y_k[tr], mean, B, seed=d), a_k[he], mean, B), k, d)
                parts[f"map_{name}"] = (m_oof, m_test)
            own = RidgePerOutput(lm.ALPHAS, FIXED["folds"], d).fit(xk, y_k)
            parts["own"] = (oof(lambda tr, he: own.refit(xk[tr], y_k[tr]).predict(xk[he]), k, d), own.predict(X["alexnet"][1]))
            stack = {}
            for name, chosen in (("stack", ["map_alexnet", "own"]), ("mixed", ["map_dinov2", "own"])):
                w = nc.stack_fit([parts[c][0] for c in chosen], y_k)
                stack[name] = nc.stack_apply(w, [parts[c][0] for c in chosen], [parts[c][1] for c in chosen], y_k.mean(0))
            add("avg_labelled_mean", k, d, nc.with_labelled_mean(avg["alexnet"][1], avg["alexnet"][0][idx].astype(np.float64), y_k))
            add("lowrank_map", k, d, parts["map_alexnet"][1])
            add("own_ridge", k, d, parts["own"][1])
            add("stack", k, d, stack["stack"])
            add("mixed", k, d, stack["mixed"])
            # descriptive: per-ROI encoding gain of the map over the average, and the who-fails forecast at this k
            pm_, pa_ = parts["map_alexnet"][1], avg["alexnet"][1]
            r_v = lambda pr: ((pr - pr.mean(0)) * (meas - meas.mean(0))).sum(0) / np.sqrt(((pr - pr.mean(0)) ** 2).sum(0) * ((meas - meas.mean(0)) ** 2).sum(0) + 1e-30)
            rv_map, rv_avg = r_v(pm_), r_v(pa_)
            for cls in FIXED["roi_classes"]:
                lab = rois[(s, cls)]
                for v in np.unique(lab[lab > 0]):
                    sel = lab == v
                    roi_rows.append({"subject": s, "k": k, "draw": d, "class": cls, "label": int(v), "n": int(sel.sum()),
                                     "map_minus_avg_r": float((rv_map[sel] - rv_avg[sel]).mean())})
            w2 = nc.stack_fit([parts["map_alexnet"][0], parts["own"][0]], y_k)
            stack_oof = nc.stack_apply(w2, [parts["map_alexnet"][0], parts["own"][0]], [parts["map_alexnet"][0], parts["own"][0]], y_k.mean(0))
            n_fc = min(200, k)
            fc_rows.append({"subject": s, "k": k, "draw": d, "forecast_top1": nc.identify_top1(stack_oof[:n_fc], y_k[:n_fc])})
        if (time.monotonic() - T0) / 3600 > FIXED["wall_hours"]:
            stopped = f"wall clock after {s} draw {d}"
            break
    pd.DataFrame(rows).to_csv(C.METRICS_DIR / "10c_scores.csv", index=False)
    pd.DataFrame(roi_rows).to_csv(C.METRICS_DIR / "10c_roi.csv", index=False)
    pd.DataFrame(fc_rows).to_csv(C.METRICS_DIR / "10c_forecast.csv", index=False)
    print(s, f"{time.monotonic() - t:.0f} s |", f"{(time.monotonic() - T0) / 60:.1f} min")
    if stopped:
        print("STOPPED:", stopped)
        break

In [ ]:
frame = pd.DataFrame(rows)
def per_subject(method, ks, metric, centred=False):
    f = frame[(frame.method == method) & (frame.k.astype(str).isin([str(k) for k in ks])) & (frame.centred == centred)]
    return f.groupby("subject")[metric].mean().reindex(SUBJECTS).to_numpy()
P = FIXED["ks_primary"]
avg_enc = per_subject("avg", [0], "enc_r")
avg_top1_c = per_subject("avg", [0], "top1", centred=True)
claims = {
    "C1_lowrank_map_minus_avg_enc_r": nc.verdict(per_subject("lowrank_map", P, "enc_r") - avg_enc, DELTA_R, FIXED["n_claims"], FIXED["alpha"]),
    "C2_stack_minus_avg_top1": nc.verdict(per_subject("stack", P, "top1") - avg_top1_c, FIXED["delta_top1"], FIXED["n_claims"], FIXED["alpha"]),
    "C3_stack_minus_own_ridge_top1": nc.verdict(per_subject("stack", P, "top1") - per_subject("own_ridge", P, "top1"), FIXED["delta_top1"], FIXED["n_claims"], FIXED["alpha"]),
}
desc = {}
for k in [0] + sorted(FIXED["ks_primary"] + FIXED["ks_descriptive"]):
    methods = ["avg"] if k == 0 else ["avg_labelled_mean", "lowrank_map", "own_ridge", "stack", "mixed"]
    desc[k] = {m: {"enc_r": float(np.nanmean(per_subject(m, [k], "enc_r"))), "top1": float(np.nanmean(per_subject(m, [k], "top1")))} for m in methods}
desc[0]["avg_centred_top1"] = float(np.mean(avg_top1_c))
desc["own_full"] = {"enc_r": float(np.mean(per_subject("own_full", ["all"], "enc_r"))), "top1": float(np.mean(per_subject("own_full", ["all"], "top1")))}
fc = pd.DataFrame(fc_rows)
f500 = fc[fc.k == 500].groupby("subject")["forecast_top1"].mean().reindex(SUBJECTS).to_numpy()
rho, pv = stats.spearmanr(f500, per_subject("stack", [500], "top1"))
roi = pd.DataFrame(roi_rows)
roi_gain = roi[roi.k.isin(P)].groupby(["class", "label"]).map_minus_avg_r.mean().round(4).to_dict()
summary = {"protocol": "P-7 (docs/proposals/nsd_confirmation_protocol.md)", "K": FIXED["K"], "common_vertices": int(V),
           "delta_r": DELTA_R, "delta_top1": FIXED["delta_top1"], "claims": claims, "descriptive": desc,
           "h11_forecast_k500": {"rho": float(rho), "p": float(pv)}, "roi_map_gain_r": {f"{c}|{l}": v for (c, l), v in roi_gain.items()},
           "stopped": stopped}
(C.METRICS_DIR / "10c_summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("10C_SUMMARY_JSON " + json.dumps(summary, default=str))